## 1. Dataset Structure Check

In [14]:
import os

def count_images(path):
    return len([
        f for f in os.listdir(path)
        if f.endswith((".jpg", ".png", ".jpeg"))
    ])

print("Dataset overview loaded from YAML:\n")
print(open("dataset.yaml").read())

train = count_images("dataset/train/images")
val = count_images("dataset/val/images")
test = count_images("dataset/test/images")

print(f"\nTrain: {train}")
print(f"Val:   {val}")
print(f"Test:  {test}")

Dataset overview loaded from YAML:

train: dataset/train/images
val: dataset/val/images
test: dataset/test/images

nc: 3

names:
  0: crack
  1: potholes
  2: wall_peeling

Train: 546
Val:   117
Test:  117


In [2]:
from ultralytics import YOLO
model = YOLO("yolo26s.pt")
model.info()
print(model.model)

YOLO26s summary: 260 layers, 10,009,784 parameters, 0 gradients, 22.8 GFLOPs
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affin

## 2. IoU + Evaluation Function

In [3]:
from ultralytics import YOLO
import numpy as np
import cv2, glob, os

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    return inter / (a1 + a2 - inter + 1e-6)


def evaluate(model_path, data_yaml, run_name):

    model = YOLO(model_path)

    metrics = model.val(
        data=data_yaml,
        split="test",
        plots=True,
        save_confusion_matrix=True
    )

    image_paths = glob.glob("dataset/test/images/*")

    all_ious = []

    for img_path in image_paths:

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        label_path = img_path.replace("images", "labels").rsplit(".",1)[0] + ".txt"

        gt_boxes = []

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    _, xc, yc, bw, bh = map(float, line.split())

                    gt_boxes.append([
                        (xc-bw/2)*w,
                        (yc-bh/2)*h,
                        (xc+bw/2)*w,
                        (yc+bh/2)*h
                    ])

        pred = model.predict(img_path, conf=0.25, verbose=False)[0]
        pred_boxes = pred.boxes.xyxy.cpu().numpy()

        for gt in gt_boxes:
            best = 0
            for pr in pred_boxes:
                best = max(best, iou(gt, pr))
            all_ious.append(best)

    return {
        "Run": run_name,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "Mean IoU": np.mean(all_ious) if all_ious else 0
    }

In [7]:
## MAKE SUBSET

import os, shutil, random

random.seed(42)

IMG_DIR = "dataset/train/images"
LBL_DIR = "dataset/train/labels"

ALL_IMAGES = sorted([
    f for f in os.listdir(IMG_DIR)
    if f.endswith((".jpg", ".png", ".jpeg"))
])

# shuffle once for reproducibility
random.shuffle(ALL_IMAGES)


def make_subset(images, size):
    return images[:size]


def build_subset_folder(images, size):
    subset_name = f"iter_{size}"
    base = f"dataset_incremental/{subset_name}"

    os.makedirs(f"{base}/images", exist_ok=True)
    os.makedirs(f"{base}/labels", exist_ok=True)

    valid = 0

    for img in images:
        label = img.rsplit(".", 1)[0] + ".txt"

        if not os.path.exists(f"{LBL_DIR}/{label}"):
            continue

        shutil.copy(f"{IMG_DIR}/{img}", f"{base}/images/{img}")
        shutil.copy(f"{LBL_DIR}/{label}", f"{base}/labels/{label}")
        valid += 1

    print(f"{subset_name}: {valid} valid pairs")

    yaml_path = f"{base}/data.yaml"

    with open(yaml_path, "w") as f:
        f.write(f"""
path: .
train: images
val: ../../dataset/val/images
test: ../../dataset/test/images

nc: 3
names: [crack, potholes, wall_peeling]
""")

    return yaml_path

## 3. Iterative Training Pipeline

In [8]:
import time
import pandas as pd
from ultralytics import YOLO

BASE_MODEL = "yolo26s.pt"

sizes = list(range(400, 501, 50))  # 400, 450, 500 ...

val_results = []
test_results = []

prev_weights = BASE_MODEL


for size in sizes:

    print(f"\n===== ITERATION {size} =====")

    subset_yaml = build_subset_folder(
        ALL_IMAGES,
        size
    )

    model = YOLO(prev_weights)

    start = time.time()

    model.train(
        data=subset_yaml,
        epochs=100,              # requested fixed epoch
        patience=20,             # early stopping
        imgsz=640,
        batch=16,

        optimizer="AdamW",
        lr0=0.0015,
        lrf=0.01,
        warmup_epochs=3,
        weight_decay=1e-4,

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        fliplr=0.5,
        flipud=0.1,
        mosaic=1.0,
        copy_paste=0.3,
        degrees=10,
        translate=0.1,

        project="runs/detect/iter_road",
        name=f"iter_{size}",
        exist_ok=True,
        save=True,
        plots=True,
        device=0,
        amp=True
    )

    end = time.time()

    best_path = f"runs/detect/iter_road/iter_{size}/weights/best.pt"

    print("\nEvaluating...")

    val_summary = evaluate(best_path, subset_yaml, f"val_{size}")
    val_summary["Time(min)"] = (end-start)/60

    test_summary = evaluate(best_path, subset_yaml, f"test_{size}")
    test_summary["Time(min)"] = (end-start)/60

    val_results.append(val_summary)
    test_results.append(test_summary)

    prev_weights = best_path


===== Training iter_400 =====

Ultralytics 8.4.56 🚀 Python-3.9.25 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_incremental/train_400.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=iter_400, nbs=64, nms=False, opset=None, optimize=False,

RuntimeError: Dataset 'dataset_incremental/train_400.yaml' error ❌ 'dataset_incremental/train_400.yaml' does not exist

## 4. Compare Validation vs Test

In [ ]:
val_df = pd.DataFrame(val_results)
test_df = pd.DataFrame(test_results)

print("\n=== VALIDATION RESULTS ===")
print(val_df.sort_values("mAP50-95", ascending=False))

print("\n=== TEST RESULTS ===")
print(test_df.sort_values("mAP50-95", ascending=False))

In [ ]:
print("\n=== VALIDATION SET RESULTS ===")
print(validation_df[[
    "Run",
    "Training Images",
    "Precision",
    "Recall",
    "mAP50",
    "mAP50-95"
]])

print("\n=== TEST SET RESULTS ===")
print(test_df[[
    "Run",
    "Training Images",
    "Precision",
    "Recall",
    "mAP50",
    "mAP50-95",
    "Mean IoU",
    "Time(min)"
]])

## 5. Plot Validation Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

for size in [400,450,500,546]:

    run_name = f"iter_{size}"

    df = pd.read_csv(
        f"runs/detect/iter_road/{run_name}/results.csv"
    )

    fig, ax = plt.subplots(
        1,
        2,
        figsize=(14,5)
    )

    for c in [
        "train/box_loss",
        "train/cls_loss",
        "train/dfl_loss"
    ]:
        ax[0].plot(
            df["epoch"],
            df[c],
            label=c
        )

    ax[0].set_title(
        f"{run_name} Training Loss"
    )
    ax[0].legend()

    for c in [
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)"
    ]:
        ax[1].plot(
            df["epoch"],
            df[c],
            label=c
        )

    ax[1].set_title(
        f"{run_name} Validation Metrics"
    )
    ax[1].legend()

    plt.tight_layout()
    plt.show()
    plt.close()

## 6. Confusion Matrix Display

In [ ]:
from IPython.display import Image, display

for size in [400,450,500,546]:

    run_name = f"iter_{size}"

    print(f"\n{run_name}\n")

    display(
        Image(
            filename=
            f"runs/detect/iter_road/{run_name}/confusion_matrix.png"
        )
    )

In [ ]:
## Compare Validation and Test Performance
comparison_df[
    [
        "Run",
        "Training Images",
        "Precision",
        "Recall",
        "mAP50",
        "mAP50-95",
        "Mean IoU",
        "Epochs Used",
        "Training Time (min)"
    ]
]

## 7. Ground Truth vs Prediction with IoU

In [ ]:
from ultralytics import YOLO
import glob
import os
import cv2
import matplotlib.pyplot as plt

model = YOLO("runs/detect/road_damage/unfrozen/weights/best.pt")

for img_path in glob.glob("dataset/test/images/*"):

    img = cv2.imread(img_path)

    if img is None:
        continue

    h, w = img.shape[:2]

    pred = model.predict(
        img_path,
        conf=0.25,
        verbose=False
    )[0]

    pred_boxes = pred.boxes.xyxy.cpu().numpy()

    label_path = (
        img_path
        .replace("images", "labels")
        .rsplit(".", 1)[0] + ".txt"
    )

    gt_boxes = []

    if os.path.exists(label_path):

        with open(label_path) as f:

            for line in f:

                _, xc, yc, bw, bh = map(float, line.split())

                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                x2 = (xc + bw/2) * w
                y2 = (yc + bh/2) * h

                gt_boxes.append([x1, y1, x2, y2])

    # Draw predictions (RED)
    for p in pred_boxes:

        x1, y1, x2, y2 = map(int, p)

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 0, 255),
            2
        )

    # Draw ground truth and IoU (GREEN)
    for gt in gt_boxes:

        x1, y1, x2, y2 = map(int, gt)

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        best_iou = 0

        for pred_box in pred_boxes:

            current_iou = iou(gt, pred_box)

            if current_iou > best_iou:
                best_iou = current_iou

        cv2.putText(
            img,
            f"IoU={best_iou:.2f}",
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 0),
            2
        )

    # Legend
    cv2.putText(
        img,
        "Green = Ground Truth",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv2.putText(
        img,
        "Red = Prediction",
        (10, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(img_path))
    plt.axis("off")
    plt.show()

In [ ]:
from ultralytics import YOLO
import glob
import cv2
import os
import matplotlib.pyplot as plt

# Best model
model = YOLO("runs/detect/road_damage/unfrozen/weights/best.pt")

# Class names from model
class_names = model.names

# Display legend once
print("Green = Ground Truth")
print("Red = Prediction")

for img_path in glob.glob("dataset/test/images/*"):

    img = cv2.imread(img_path)

    if img is None:
        continue

    h, w = img.shape[:2]

    # Predictions
    pred = model.predict(
        img_path,
        conf=0.25,
        verbose=False
    )[0]

    pred_boxes = pred.boxes.xyxy.cpu().numpy()
    pred_classes = pred.boxes.cls.cpu().numpy()

    # Ground truth labels
    label_path = (
        img_path
        .replace("images", "labels")
        .rsplit(".", 1)[0] + ".txt"
    )

    gt_boxes = []
    gt_classes = []

    if os.path.exists(label_path):

        with open(label_path, "r") as f:

            for line in f:

                cls, xc, yc, bw, bh = map(float, line.split())

                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                x2 = (xc + bw/2) * w
                y2 = (yc + bh/2) * h

                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(int(cls))

    # Draw predictions (RED)
    for box, cls_id in zip(pred_boxes, pred_classes):

        x1, y1, x2, y2 = map(int, box)

        class_name = class_names[int(cls_id)]

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 0, 255),
            2
        )

        cv2.putText(
            img,
            f"Pred: {class_name}",
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

    # Draw ground truth (GREEN) and IoU
    for gt_box, gt_cls in zip(gt_boxes, gt_classes):

        x1, y1, x2, y2 = map(int, gt_box)

        gt_name = class_names[gt_cls]

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            img,
            f"GT: {gt_name}",
            (x1, y2 + 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

        # Find best IoU prediction
        best_iou = 0
        best_pred_class = "None"

        for pred_box, pred_cls in zip(pred_boxes, pred_classes):

            current_iou = iou(gt_box, pred_box)

            if current_iou > best_iou:
                best_iou = current_iou
                best_pred_class = class_names[int(pred_cls)]

        cv2.putText(
            img,
            f"IoU={best_iou:.2f}",
            (x1, y1 - 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 0),
            2
        )

        cv2.putText(
            img,
            f"Matched Pred: {best_pred_class}",
            (x1, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 0),
            2
        )

    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Green = Ground Truth | Red = Prediction")
    plt.axis("off")
    plt.show()